<a href="https://colab.research.google.com/github/Pranav5435/Image-Detection-Model-/blob/main/New_Trash_Detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.1 MB/s eta 0:00:00


In [2]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write('{"username":"pranavkishore","key":"KGAT_7826dfdd2cfca3858007856eab98ffed"}')
os.chmod("/root/.kaggle/kaggle.json", 0o600)

os.system("kaggle datasets download -d mirodiljondusmatov/trash-detection-yolo -p /content/data/ --unzip")
print("Dataset ready!")

# --------------------
# TRAIN
# --------------------
model = YOLO("yolov8n.pt")
model.train(
    data="/content/data/data.yaml",
    epochs=20,              # changed to 20
    imgsz=416,
    batch=32,
    device=0,
    patience=5,
    project="/content/trash_detection",
    name="run1"
)

# --------------------
# TEST ON IMAGE
# --------------------
best_model = YOLO("/content/trash_detection/run1/weights/best.pt")

def detect_trash(image_path):
    results = best_model(image_path, conf=0.25)
    res_plotted = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(res_plotted[:, :, ::-1])
    plt.axis("off")
    plt.title("Trash Detection", fontsize=14)
    plt.show()

    boxes = results[0].boxes
    if len(boxes) == 0:
        print("No trash detected.")
    else:
        print(f"Found {len(boxes)} object(s):")
        for box in boxes:
            label = best_model.names[int(box.cls)]
            conf  = box.conf.item() * 100
            print(f"  - {label}: {conf:.2f}%")

detect_trash("/content/your_image.jpg")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset ready!
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=4

FileNotFoundError: /content/your_image.jpg does not exist

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Retrain with a bigger model + more epochs
model = YOLO("yolov8s.pt")  # small instead of nano — better accuracy
model.train(
    data="/content/data/data.yaml",
    epochs=50,
    imgsz=640,              # bigger image size = more detail
    batch=16,
    device=0,
    patience=10,
    project="/content/trash_detection",
    name="run2"
)

# Test with lower confidence threshold to catch more objects
best_model = YOLO("/content/trash_detection/run2/weights/best.pt")

def detect_trash(image_path):
    results = best_model(image_path, conf=0.1)  # lowered from 0.25 to 0.1
    res_plotted = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(res_plotted[:, :, ::-1])
    plt.axis("off")
    plt.title("Trash Detection", fontsize=14)
    plt.show()

    boxes = results[0].boxes
    if len(boxes) == 0:
        print("No trash detected.")
    else:
        print(f"Found {len(boxes)} object(s):")
        for box in boxes:
            label = best_model.names[int(box.cls)]
            conf  = box.conf.item() * 100
            print(f"  - {label}: {conf:.2f}%")

detect_trash("/content/test_trash.jpg")